# CropCop Prompt 1 — R07 foundation, runtime export, PTQ and deployment locks

This notebook executes Prompt 1 only. It clones the research code, consumes the two attached Kaggle datasets, verifies all frozen identities, and embeds the plan and P0 audit. It never opens the V1 test surface.

Recommended Kaggle settings: GPU Tesla T4, Internet On, and a saved notebook version for immutable logs.


In [ ]:
# Prompt 1 configuration. Attach the two Kaggle datasets listed in the runbook; no ZIP is required.
CLEAN_START = False
RESUME_COMPLETED_STAGES = True
ENABLE_SINGLE_ROUTE_PTQ = True
NUM_WORKERS = 2
SOURCE_BATCH_SIZE = 16


## Bootstrap and attached-input verification

This cell fetches the immutable research base commit, applies the embedded Prompt 1 overlay, verifies the two attached Kaggle datasets by frozen hashes, and prepares an isolated pinned Python environment. No input ZIP or dataset download is required.


In [ ]:
# Standalone bootstrap: clone the research code and consume attached Kaggle inputs.
import base64, gzip, hashlib, os, shutil, subprocess, sys
from pathlib import Path
RESEARCH_REPO_URL = "https://github.com/rana-m-ahmed/ResearchWork-CropCop.git"
RESEARCH_BASE_COMMIT = "a4f9d6e3d417b024ecbfbf4247cd847881935236"
FINAL_V1_DATASET_SLUG = "ranamuhammadahmed6/cropcop-finalized-v8-11-2026-1"
CHECKPOINT_DATASET_SLUG = "sabahatabbas/sec-je-r07-cnxtt-context-s1-8904b100d223-a01"
BOOTSTRAP_ROOT = Path("/kaggle/working/cropcop_prompt1_bootstrap")
INPUT_ROOT = Path("/kaggle/working/cropcop_prompt1_input")
WORK_ROOT = Path("/kaggle/working/cropcop_prompt1")
RESEARCH_ROOT = INPUT_ROOT / "research_repo"
PIPELINE = RESEARCH_ROOT / "journal_extension/scripts/run_prompt1_kaggle.py"
_EXECUTOR_B64 = "..."
_PLAN_B64 = "..."
_BASELINE_B64 = "H4sIAF9cs2oC/6Vb6XLcRpL+z6dAxPwYW9NA42oc0ngimpfUtkTRJO2Z2SMaBaBAwkQDvQCaFB36sQ+xT7hPsl9mFQ5SGo93N0JHN1BHVp5fZmX/wbiyQ+OybfJD1pdNbRyLTlZlLY31IS/7o6NXr96KXr5+9cpILm3jcn19bfz3f/6X8eHj8eb9mXG8vj57v7k4M86vPv7L2UViGJjAM41cT3NtNzDt2HRd45t1V4rlD6IV2V35LY/90OQ8qpUiN5u6ejLEfl+VmSBill1zaDOJl1lTd32rSHxj1I2RVaLryqKUrdG0NMfQYx9FZ2R3or6VOW/w49qsmuwe3/aVqI3rd2vTXQV8nswV3srPUruI5cr2vNB3c5niUeoUme+mfuxEjoyiyC9WaZAXuXT83M1XthOJVZzZydHRH/5gXMl905V90z4ZZS7rvuyfjo7M2ePXRnK9Pl6/W9+Y62MwbHnSNvuTZm9+aNKykgkGK46lraizOwwvmkOdKw60dmju22a37x2TOGnHrkszBjEtD3swRoodpjVteVvWy50o6/kQI2t2u7LHgFUW+iA88oUofDuLHNvOizRdyTBN08DNfCf20zz2ns3G6pqrlQQDU1k0rTTKuttLFoeFwTd30jh0sv1jZ+xbacpPZdeX9S0EIbP75tAbojeS09f/hpP8glndv71gAa9fN72xa3ISam4Zm54fQiOTHvpyb2bLDGI2bU+YuXwoM2n+x0FUGK10JeE94iJPIccw8/M8smUg/VS6TlzEIneKIvVtJ/TxX5EsjMeyvzN6EP6M4kPNu0FdkpP15eXW9tbby3d/v96crN9vT89+3pycba9+urjZfDjb/vjT+v3mHG9uNh8vtldnlx+vbqxdnljPxA/21z1E0hlRaBvD6nvR33VEMlEgSPx4qARF0xP5SWYHTM/utkqxE6PEeOO27CGVez6sdAsnkqnrZf4qzV03E57npnYaQXNF5OTSs/3UD10cNoUMYDWJhfng8aGSXWLswE86Mx++s4wEL43ukKoBRteL/oBhoLBVQi9EWWHczR1IwR8MqKCA9aETaUV8bB5kDRWWC5Ylnwwq8gBfUudtU+ZGLveyhpFkT0YLtZDWSwvqm31TNbdsQefVoe9lO3cI0GF865a70XKu70QraeGuvK2N7qnrJZnCHlwWt7JbZlC0rNlv1YCtGkATL8tbCfuqBRNY1gUOCdJhhGV+K7+2hCi36iWbR9vcgzboPKkihutP3ZKH0UsaBk7hXKC+gvbWTU2fDEU96brEF7jJthQVluAH3fKh7EifMfsB+q38wHzY9JTGnGxeK7HeHdLlY9PeF1Xz2C1bCWvt5DY9lFVuPe0q5axumqaCdyxrQ9T5KJbJb32G3cmd8Vl5JDAWmx2k8fnos2ma/BdDBsF8NhLP8kPLgYJBF6ADC6NoxU4SGUYSBKuVH7kSjpIWME5F26s5Dv6oZ++arje+Fw+CDP1yffMOAz6U4HjXFL3xEdry/ekPRuKElm05ep1h+4F6nv55/Hrdg/TG+P74ykhcZzZvHIAViYzAwluQjrjQQ713S+aV2YNFnXqt5r1tRQ5xPbZQPWybS8SeVkkFy8QWHebN+LrL2nLfd8vv11cGVFNZgjZ64jk9u5W1xAp4kj5N3CQSEcfWby/VukTd+PSHhuwetB7g4um9a/nD+2PWymFiZLkD5frFF3MdK7YQRnjMCWILtHHZi/ZW9ssdRigGUUzNZSEOVd9BrM1uoPQ1MWeJP66Pc4/24vlL/MEzXvaC1xjOptcB1UydF4SBt4rUSBLfUpOoiMDE73/+YDjhcPrB/I3NqbE0amhYB+uUdBS4TEt+Ert9Ja3BULV3GHn3INtOS4v0wf4T2PNNop9eYLXv+DFUYXh4Akv8zkm+5SWulCkZ5EDIXX6GwySS4ImfKOpBWfSrk6YuytvOAoo43E77r483BtxaCa/3mb2wSMvzsgJjOmIgaR27flKPnXgyJIIA9GQX+OZDJBb0UWKK+RAKVqFPUbANfJgmHEIN4TC/lR+bwaLusN83sLhxZyUpAw4zN9mTI9ghDoD8cS8OMoiHHfwBL5vsDykFeoswVDKgJ+Jk4YdeuFoFXhy7WRA7cezHuYxdOy4K6TtFAKCUpp7nBGkQCN/xchuxScRuLCOMXhF7jgg3aE+l+dDKPWNR2IZWHmCKR9Ey5/s7CnsZJNaKrVDmvFVfP5H0vvTUC2ZYUqiltuUOHn0LXtMZO/qwI+SA0F9VHGeeOGppbdSGf6nshqVzgLvalbdkvAZ8hlEc+gMM+hi09yYG6Zm0AwQCNYRX1UEOyKLcyXkIpBV1/KlKgL+2lB0FvrUKmlpe0wSGeLfW/olxwesJIpiaF68ddkbKWj8IRGMFRbqmegBD1+ur10a4cMJw4fkRfE8vu8Uo00TEIsuAd1ciLAAhVpkdSALHUngxYEYWiJXI/dDPV54bRplrryIMsVPhF16YO2HybLPLjx9GsA2oEgRBka0AuAVA6CoSeREUTur4aRis/DgMxSpwCoA0L/Zd4Lc8CvxASttJvShwvfz52lowGqdMu4RQPZHGgeN7xWoV2WlWxDmWiT03iIOVt8rjXKTuKveDIM9dz/djD3DYwQbwUGnBu4BLg1QIucESutdHhmHOzGQJcU3ct7omeW1Ei9B1F17wJV9XYRCDLBnKLA+yrPBcGEgqojyUgKSxLHw3EnEhnJWbhi7OEBdeHnmB4zpe5K4kqOLtldl/bW/HWcAOF7Hvf7F5CGjvCC+KPDuMV6ET50XoRxGeRkCNueM7KYzVcaUnoiAKvZUd+K7vZ74deo4dBcySi2Zg+Ux74SpH8G3sZC+ASQQlY7Mx5DUKwjoaYmKtM6L9hmgnb8wxjIBt1TWj46FAACvcUZZI6Qv+lwrClipLQGaCLEKFzzl+1ikg2ZUeKQ79HfBX/8TW2hNwTXVio8zyA4MwmkHCRpzGnK6Tfaeck8avrTZeTKf9JlRnqATmAkFrc3ETLf92cXG5PvmB4b1CwUivLcJVa1p1RFrwoO8bxTv4b+OYpIb/R/864q3X4z/kjmcbX96ckRueweGlInz5DEZu+Zu17yVCzXOE+TtG6DRtq065pSNu9RG3l1cfT3864dTnweHpoMf1FrC0hbdyDQmxEYVhFtq5XUD/CjuHCkbQq8C2Y8fL0qgonNBJ3dgX0s+lHTleEWa5k8PXeIEbpXmhoczs5JSvIfWDoiAt0aj98z+h/LKVP9KUOdVXZ+dnV2cXSOiYfiLf8xeRvVr49opWhGXIEI4LfiRzbN+NY8eNXCFzVwrfDsJUxH4Qho6DIT7yZ1fmIhBR4cjUtjNbozMOyZRraUzFUlJh6TnghwbuSTG8heuGI/v8oAgjxwndIHNiqkrkIfxC6uaRE0apE8MpOyDKgSP1igAB1pfSzWy5KoIginKN8oZcoy4L2fW/l5LA9Uc64tArZCztoghEnhaFn6a+X0QiiEN7hdxaBH4snVh6q7hI8zwIsjBPY/jVrMjt1NVwbM/xHWlSx+WBweR+Jz3RauJLkXuBDOC4/TxduXYepKtc2IWfwfH7ULfIzrIod1yEkNgPbScLHTi2woNEfdtOtXBubtb/ayqcRbSa+IKA4vurAIElhi+309BPnTDIvdzNXUdmYE0au6tVHBKddhi6UZEVUYG4g4AY5LFWEsEIQCcVv4uMOJxsTCIe26ENTOy6Nhw7go2fZlBGN5Ue5BIUtliloee5buSsViJYZQBkmfTsXOaYN4KxKUmd1JbdM9XnngydEyOL6eDSiWfvz80auRPI/5X9MfzYa6MVj+qj8Z3WZjoHPR0cnKTajXKmACu/SqTuWUmuERFFbW3S1neiu7O4utRUOTDtydVsP8WtsusOg2PWZQpyu0Ms4QoL3laMqCbXf0OjORnr76jAReSayjsP3hV8BiAbopGOLVQNs4wN4GCOmAMCFuyDVe5o3XKUtO77LqEBxLjKaIovRAj2iXpMSJWo87JFUKMiyOMdhcyx/MGhURcjfsPhUxLeMKnZoW3p7GqSNjQkJod2YBkO1bdl3dFZgXnzg06xHlTwFNh4hyTW6MedwdTaHMIghfQOCQaUTygher4VGR+OZ745IZdqDRB/LPDwacaK2KyWsdwLitNjtSV7yirpbm9J8DU8dEIlwW5WZBrBwKGA0pDycIy/4WLhiQIBjBhm2Zr8lFUHqLxCyJdcVzWc5TBHPpAGZFzLVPBAVYnUYmOJiII74fRrpRId4MlO/GbFaLlX1SYdccW+tHLR9gyu3o7FACqQ/PYycBLLrs2WYwFhvuLt19ZUCclvr6qzB16ZisfLe560RBYzjF5Oo3/mHde84b3ajqs460uqRFWNyNUIRlaUlUEn8jLrN5R70Xed8M1HcclTPbhUo9lMILmG8mvyHFVzW/YdS6Jv9ua9AoSsTqSsdFxkYsu5I6WxKpvsTZ0Fjzh1sGokj1DPbg8cJtnTtFqxADL3VOaRUBuqOLD/MpmKua/CgndNzvTzUd7j/MhTDlU/kc/XGrAWCnsLHYppyYV2sllzqJWrl21LMFV+6t+Q8uaNVDquVoK7wqbwukttiMspnC4np6kxJvmuFn62bCegTHc9Q7iDgn9MqWopRy6xZr8+OkqShGg4OuHU+q+6qHhCE5uqgk/6RiXdhPZvBT154qnfHhnmX4zTUtzWsPfuI0A+0AaY2LT8hiBvpfUHrqvrdqD0WhVOeYB6txksbf5KW6IaoQpbpHbQQ36t8d66VK9Uzs5vdC4HIALGzzjGL2fZyAfOJi34kEfR5vx20rt/BZD4d+NPw2KsgcuBa4NS8ZxBAWGbBC6GMdieKddjhih7XzePlaQ0v2qa+8NeMRCJFWRBpaiuJz6QQNgdqSRiqAUPIZCDq3bNZH9HR6e63sYlV33VAMVO/kmGQCVCjMx/AZCARo4qtXiO2xYMm1S1SgNKtTBUStehYOScqzPB51SNg1o5FsInqSVin0jx/tArZ/rmyB1ejVEPq6uI+CJMjtW/7s2RZw1BmoHQDJ0wduqbnQ5wiPHN4RYZ5585dJ2W7V+Wf+ZRl9heXQV1OO0d28y+EpmymQPy2LbLACq6v1hdT4snb458S+WFghDO/2fJhI/5iLinmIKQ2INekyW5E+29bN8crSyjwENKXLN7isiwTNnWoloSTFShWHs6ulsz4V6VDFjuEMk5Z75yLMLOaoR8/UXk7oly8qzsxsgOSJGSl7iKgFf1RB7iz+PmfHIlf6VK3fYLxdLgkmSlQSLr44AG+ZqQ5HZLqKDXcGk7aJf1S0fRQN88PiNQgUOFzZa5pDs8ZRMMMCnRZrCjNIkAQI01SBKDRhAH4F1VOWFUZ4U7z8u2601gAYbcy5dmV859M0NiMQ8aOwS5A/leWef7BoBLIYr32IGxDVf6wMSyhq+eViU0wfdzmgvjeKAVgietNU6Q33wLDSp6LciCyB0NiC9iKC6xD6iGVfhWjzB6kgFZUTgmPsKoVWQGyKcPh1o8iLKiKx36CuLlLaGzczzEmZLhzldfehnkFSrZg3w4igcqmSE0YFXRTuBSKvWGO0MQVBCXweoUVW4PcLwUDusBwb7gDRtVsn3+VFVLVZ1KKb6KsPpcL7AbSVNoek2C/gIC0uW02RJDUTpjyxtUjhSsbZ6Y7d1TnUGH6ubQQdjNeL2SIxjAFzXj3SDcGOBFJyrNNJ121M0ITAF9zq4253/fXLwlbl+dvT9bX6svGfGrqvigZicKOUQgRTPnHwjN5k7ugK4XtC25iduWOgiWZC/qoxY74xSNx/gUOLZJjFIoXjGuLFTpTOd89FrmSnlPOYgujLO/bc5fBIXnfGblHi99aVKnpXdc9ogo54JTHeUA+YKXFP7sU1lsoGstQA6iUdMSzBJDk8G1utCARdH2dKlWd3RvRwbOsN5om16X1GJ76UT20g1t3oFqgL9S0lEtH2SrUryiKvfdG7UMWfCSP9EFkBwv7mapwx3WqWQ+aP7zetSMDR0eScXOqqlvodYc3uFZydOxjS7obT0eW+DpJzkrQO4FfakaFQUpFIMoXhFBQ+XZzESiZaNR4qtXoPfVK6OkmzCCVYz1Tm5OzAf3jzMgCFRh8i2IcfX2GGqDqNPBVbYUB3K2F+1m1Va67I5P6qZLpZc4+o4kmR3SMsNrvn5rlVeQdQfC5/065CiBH0XvucbFybu/GonzyfsEbtBf6Hhekl/j4OiuVnxmzhUuZD+rLTAlGuQwKTMDNAk8Ee18FZw8R3LfnP1Md9gWpXzflPX+0Csav/0We5c7IPtSO+zRv5MuEqBbd+dE97ptxZO+Svry9oxY0hx6rAsNz6HiOC7QosU3q3CK7NSRIFJlocP8B6VaqZQU8IdiOamY7meh9BQs5MIxGAqo6SwJcHqslp0yAeUGMIkz2QmoznwYWd2Qb+9AHolvypRMhJLygcOTbG+nzOiNDhjsFLhuwCkxol9LzUWSvPWeEZXs7hBCdErGfTQK6lbiCVoPQMrzVZUEZtI/NkuutZgPpXwkCGkZa7p2a59ZE/sgc2TLaBXdeHtO3GulbjpB1CVFH+v8yk+djMUrmjSC7KMZc8YRQ5ygwNWOOj4idARB0AM3SbL6ZHx//fGCqieNwmt4RO94kKFN8h3AAy046m7HdavBsPhusC07ct2iAtkDDBmC3oSPBhc8UfP+nHb4YcwaqLLF/pmshwXH+jAA/fHk06ctoWSqeCtQxfyRT2qBl9W/zeng7mZ5CkuXHCNXcIhwBff0nNnlCx9qBJtaMipxpFE6cySh6Kds1wOA0fe8yQTe/1E6avXiXl4ipEGtEdW4AKKzUnU9r65992VGXTnPoB/n/xMEoanHDeFhdQnNvrnSGJlvn6ntiYTUN9P8fEh4l82Y8Q55mQ7Xz2OE7n1SrCX+EC5i1DmDkGNmLXpCFOzTyfqeVOWjolx62BquZT5MwN6esCpCbyVvOU2Z06vlpfGJLnRN2SbTrZYfaSMhqbRUVbuGnorXhscYc+AAjvaoOwhaSgNHvflZjU9mFW39aJNzD86BvQV42x2AD1W5hhNNqjqqQIh/qFgoR181lV30Wl0y1RZ5j7FSQqgIHmUqkyyfiYRy9We1o/GiDqoLJ6d4zlnUeNRmVt4YjBcOWadzfE9IwYLccD73cJtTI1EiMJEbmcP9mDnb3rStKLKDYBWFnrqCvQZjSlYltdLTctIbfU2tlB0SbJ/2PbcYPikvTszrABcpemAqBX+k3+xViWe0/Pvn4gabmkOurFwnVOSP8uaxVt2mVNRWZ0Cc47EErIkpe+Xu38wY1codS4R1SqnbgA0RjXZlx6JbMHI9KJddy56byJQekPZ9oKa1MUEjPIP/O+rS/HB2tdZ6tbm4Obu6OLuhm8yrs5OPV6fb9U+nm4/JizowKGoeFKbQrodE9SjaZ2fQV98TTB/aLGCYzeMxU0uAfXhMKnn2aeDrFTdZzl4Xh6pSk06UwOYvSXYnyDlaKrzdtIJq2gnj+XGI5sk1SRKaqZqNEuWRZpfmMzZZxqXubZs6uaZ0nvtd9Bn3YwO4Se0+qrSpV1kWlXgAIVNr0NC2OW/c4RKEvisQqnX8lDvloIknm6G/aVZc1D05w/T9vWGaQ9plggpk7yahVZGWqv7FJQTVgzz0Jj/yPZRss5IycEow/m8b4HvdmPtD+k8mqf40c+gYNLRoTO4MmRZhctfqEsVwuG2WCgS6TKZ7Tk3uEjO5Gg0VBNtSmQkyI5Jjcvo6MR6a6rCjXO8Oz3u+tpCwYe59Uxn0eHODqKKyN12J45akfACmGWx23i+hSRu7ujm/Gwh2DUoQW1US4do1zSlhzFxJ5nWnyypCQowr4VjwNTkB3b8caoX88VKZmlqiLlqh0gICoOocBDEpQ+WO75bqeFK3dGbcmTiFUs3GoUOsTEHZwLKpo3IQFjLfcozEY6MAAXv5YGkpWypn7yxV4OCx254MZzM9uMF3VU5OuG6RMcKfVUgo+e9njWwcYggxUAqAiAZPxYEBGf/68oflen2sQJrq9h678i3LSijG4Uy0RElXDmR1qtF+LDCpawT4kskKtNlBhNcDdsBGnWq6kfngGnSDfkVpznhNNnTojwEOAUyObatKa7DxYNnjDdkX7TRj8zq3GbNvJqGdbBjfbYhncADIcIxk6FLeUmVE9NldwmpmqYEwSZXVD117BkwKAiZ32XGCNkJEEhXixmTK1Gk5iAEsUMoKmG2ENt1OKjPlbJbPrVIKYLQWsAWxsYUZdSadiANmKwnuiFr7SNA2ksQI61cqjmmAd6ghSqIH3H7QnT2Lf9QutRj5SY2jnI69eE9Koom/Pv74YX4ZsKDUHKSSTijvQTRyqJjwO+VEI4SZtZFPfVgnOtsYaMZn/gmLKqV0huerfILfc6F3YaymjvHJPsz5iBnk1e2LI2OoFM3XLeyIBq35cf36SHUhc/srddvS5dmLjvMv+I6B9FMkarFSHQCqC3YcxzQN/hgfxppi9vSdM812fFvRxrOVejzvR/1snK837xdfuJV599Fzb5B83Xco+uj2MC+LgkgiDUymcgs36yrCqBsDVvQzq8ScWyn/lKlVpjVcKc2qO2zbfNFySRcOd6prQjzvoNZwzKJLlyGGE4iAKuogqq8LEJ7GpudnHQQW3bwQMhq8QTl4wPGeX/IPAkq+/hBkwaCAOg2IeWXKOnDYE5hfDjvQAcmLqcIylToHng8HNbh68/xOIrn86fj95mT70+Xp+uZsS79FW7892158vNmeXd+s8e763dkp8KC+7HdVD6++ntIuTCfm9KsxomQwnIE0i2583j1vvZuS9LG+netWEso3tCmIlw0R6jpCNyEMiaiArQCov7jo0g0nY6S16Cboq52YpBSUw3y9I3Mw+cDi6wYiKx1/MDA03ut0Tp0IBOIjuxt1LT1UJga/S/3YGJrSL7sowQQNuvr9w9vLubpZR6GS59d+zzT8lIlTqf2dQiGR9QJX4u2UW/EPa16WzZCKkLtTbdl09AEUJC/DOrNfWbBsOTDqtHjQNMYdww8JS6rNgAEzPRxYbh3F1uhBvyyDU9fPLJvj5FIgbu/5+kP9pqZ7WfgaO0knBJ7KO/FQDnVgUau6IciG+pcaz3GZ0LHnt1VfuajacYWZz0s54fOVx67a8e6q4cYAcmdYG95EpS1L0idTjhnOPFMzO52WDL83eFlCV8DgmH/ZqIoa5CI4AT1S8XUqFS1f/J5z+FUgbI1CtPHXS9O2dVQmVT0QVvrZUV6/O/D1gfr1IASmYJTUkUfPSqa+IqOkXzzR6IO+xnvZDqTnqMfHhKmBdeQ0peDLnmr6OSQGc1HyK+0aypMt1S8YJ8/GpxQ5bzV6Ks6Ah19Gglkt0380/A43oStGXYKrVJFp/BXvGDFUG1TH4BkwQ2mdbrmDSh6UwJ93m/CtAK06rsJmw0ZgckXSOvof1uGDwDg8AAA="
_AUTHORITY_BINDING_B64 = "H4sIAAAAAAACCrVY224bRxJ9z1cM9LQLZKy+XxLsA0PTGyWOFEh04t3FYtCXamlscoaZGTJRgvx7qodXybJgY7GGQWiG3dVdp6pOneIfXxTFWR/uYOmqDXR93TZnXxVn9AU5+zJ/5esm1s1tVcf8en49mX7/zbS6JrqavJl/e3V9Mf9X9c3F5cuLy39WG7rd0w9uWPd5/Y+Tm5vtu1s3QH7zihT5ZVEW0zfX17PLeYG2ipdXP1/ezK9nkx+Kg9nip9n1xauL2cutgdABmojVegjZDiNMlcSWjM2p/YoQ/P/v7cJfXLVow3tculq4purvHJMqbwnMcSmCJ8mCJJxrwSJ4fOVpCoJ5YamhYIwRSXoVUwQqIouSUOOkDTtAOli1fT203T3a/APf4LvGLUfnOte4clm6uyXE82vowXXh7ue2e19Ou3Y1bVejCdzg1sNd29XDfeVxT7jLm/H5tm7OU/3b+dC58N6XG1mmunGL8s51ERoMQ5ndJpaRDw2Fdrmsh2zIJOKcSNqLKAiNzEYdCQVQMkopmfWKe4VO7m3sQ3y8SmrXTXQD5sJ5R3S56trlaqD7w9kZ7vtzBANSgjDUG6gOFzmicrzbNnlmk8lF+d2s3CVRiYEvj4EvN3x/oWP+TKbzi59m1Y/Xs2r2djZ9M7+4ujysggWejWHuQw3NUKc6VMkt60W+whka/2Dh9tsqQh+6ejXsUj3n33kxbZvNJbwdynnd3OenAX4b9hbGeFQBd64W7f0ST6swDToMcINXRfd3dsob+ngL/LaCrh63bFHIy6aXb+fzcnp1OZ+9nZ9s2t4U71V519cjAqlrf4emmGdz5aSIsIFFu8r2CtjUmBUBCg+p7aBwePHtum+Kttv9OS0wesPof9Fi7bSYqQ+u6NGVfr0Y+mrpMIvuXHMLH3qX3KKHR6598r5Dumx3uios2n7dwTFVjsmriHAuRUmBEdCEWBNAuKSEBkuCSUlK4yyYvRPo+dItqico59F5kNEfuvXei9j+2vQDksqywrvHNqV9Dv/+eCnW8SLTCcbRdfdPJ9lYqGgh1kOFYUxotB9riQYdrUlOaEeZAwXOSQ40KLDUC3yGxCIR3muqA7KNCdoBA48fXuKyY7Gd5HkO/FAPNfRHDLFgXQ8DhqOpE/TDCfP56A1jlIYQGJGWSq6ZAHBUG0eVIcZypzw3gN8LnhwII7xzlkaflAG2dzIsXJ/jvTqxLVTShlLNVKA202jUjHLPokHrnlonHTXMUa8VT4o4iwezQEAmpYyJR9tDqDYMaaxJ9e2Jfckt105FA0ZHaryOQIyCEGkghGCGRCuU1iRIxpHWvTcyJMoEEUYrEcTePhJZ1VNMVAjvV23dnOITg07gGJjoMSgJ2wFhPHgJ2jLPkC21khYiBMaAEqcDjUJ5JFcSPbf68RndutkVO7LdU/VeGosRp3h3xng5IfQJAx2EtosnlyRJEE4xSCA4diwCQomEdJyzJIlkiKOccR6sDwYimtdgVEhaCBWkpOpxUWCtLrByh3YBmfhzNZIXJP+ju5XNelmNIR/TjDLymFCxFY4Ng+n9N8cU7dt1F+C0LZ34jC5Qq7Wi1gajXQYxItCKELzHIeH3DWQkE0y7AZnULdDWf8bTtnmfUUMXxgbvmrapA9bhhh8bUtX6d3jdnfu4fOWGsTLf4Q1z0SLRQ5OFzzm28SZmbu3Pd+SILIchOeGKDX/xrsfGcbB2DBACnnJFk4AlHy0IESiVWkZUEY6CBYuKgjrlNHcoQQTRoChJkkgiAgvsaPMWScQvWp+jT0dWVIQ5p4jEPFQa+3rShGsMe8AqlkCwBLG+mDgbLfz55dP4LB406gpZD32HsB47zsYtaqSQtvsUoPounAeUNKFdVe9gp1cyVC9W909BI6OVknOrIk06Giu51wacMVIHbpwAL5WNzjAsPyJocAqERf0ic+FrI56BRkpMd2SFJIPFgkRqkUhjmkgXJEfZ5oThRhL3PDRh3XW5R58k8BGaLCc/BZVDQyX6fCtyRqF8UC7V66vp99VGfDSDnBfUaAZJ4gtrNQgIHqKzSjqDotRpahFEFoVARZqQhqxDkaokOsupTs/AFKiUArkjJC4pdhkvGHAVFD4SnoRiiaH2xX72STCFNmImDQPkvlufOvNZ6EyvXs6qyXw+u5lPRoCewYYwZkQkPJMH8n9iSmGzAe0V0IigYB0pjgTpVMDUscmjUsD6MJw7bBEAz2DjqQJPBRao1y4YBlFpSZFLAekdiMXPyMGR57HZKZGD5kDN3MFddh6rbq94PgGnvaw7X639og473Nz5VmHsDFVbLU7tFsoJQvkDquRvZ5c3WTBPX1/dvLmefRTMKJhM1BjMKIIdBWnZYCop4zHHLPdS5DmJU5BCS+lpAE+4t96T3CTtQfU8BSaVKNkoFTpg4UnLOJKfYDZQDJjmXBicQHAoUZ8F5om6+v9B+OricvJ6Dx2Oty8v5h8FUBmeksfZUDJsvBZzBAsViZlRJCxF0VlOKY8UO3SeJFVCdeQp0wFJLTGwzwCIooYljtEA75UHsDJHghNkBccZBkYayaPfA4if/x27Jer8DTS7Tr7ThKd6fDvtzO+g2PDi0AWKui9cgWWMA9+ibXCK2PfAYts3C1Rk/RoH2sLfF0fOf3HC+bvWAad98ngAWi58HiizgedosVgt1n3xDDWczIcLqDY0a6TtrwJPTih+y+YoQpbr4YlVd3WP7W7UC5nh+8fLDjrkMEFVmDfJhYfUdwAbLzSMuvswi+Rc2v1+cfK9CwFQU+E82nbVbvj+iAPHk/fD1nq/8eGoVR1Gx48Ma/+7pZ5VuLjnCILvcfhYf4goetLuZiDEHXtEN+r58aRf6+EuZ8sH6O6y5wGUR5FylCW5/Xx++u3Bf9jNP8vsk0Lg7HEL/CSTKGDT/ce659ngulvIIcrfZCure/SiKcplgX/huyJ/oDbNWfRAcxXlL08a2QV7Pxd/VXBdrLKgj0eVfVTNh7m1Gn8QHIl4S3l/Oywqvru5uixyMItHG7crixblPh7/dfFm/qo0Xxd92+XfHt7DfY8PsHJdxqgv8mDgzkOLpPN1gbydGdlhwdb/GNPj72dPXe5kpBXUCk4cT9QCNVwAxXHPOIOdWkTuHY5oOMnhSI3aGjWiF4prfBYGRZCz7OyLP7/4C1ndrof8FAAA"
_RECONSTRUCTION_CONTRACT_B64 = "H4sIAAAAAAACCnVWWXPbyBF+969A8cVJStTOPYN9U2R6rcSmXBLtdZLaQs3RECeLg8Ehmd7a/54ekKAYr8IqUsL0Nf1199f47VWWLXq/hdoWj9D1sW0WP2YLekkWF0nk22borB+KGNL5HdHLN6uP72//8WG13izvVte36/vN3afrzc3tevlID0b9YIexT/pv727/uVofTus2QIWHv+EDPsLXHXSxhubc9/X6y2azRKeb1ZfN8v7gD5VLW8dqf9SaD23nt3EAP4wdJNHQ4sFjTDlcTsH6S7z+YwNfh2KIzX62w8N+6EY/HJI93AfPnyA+bId072asqov5uBnrwle27yGJKCOT4PfZ2Rb8r7s2Yhr91jKp0k2C1yVYBia4nLqSlp4w7p0EnTPHuFNayRwCeMaAEqs9DUI5QSgJjudaLP7o3O2HKT7nkhGmFX8hPsIOxa8w4dQPY0BsZ09Va8NRIUSfdDv8g4oIBBx1eqgQTAgF7Fq/RRnTRwm2QGxi81Dg7xBtFb/ZBF7RtMOE/DXCvIYvQ7FBmIufDzhe3ny4+mm1Xm3o34vPNIt9NrvJdl37CI1tPGRtU+0z24SsHvshQ4eZgyy0T026MYSs7bIO7G5XRXwIY5fMO9hVdv8DdlDbDZeLV8dqYB9HTDmW0WNLpf+G/XO/9e3YeSge4lD4tq5jSn5hciIcJSQwxkFwmmutaJ57o20qRMBiKUI8zDC2XXyIja2KbmyKDnzbhbOyk1IQTrkBdOVoSUAoURIsuaS+FKUhlnLGuc+dNxAwsAajfKmFUF5KquYowQ62h6GobRNL6M87ywVnGKPUe8+IzKnkmgkAS7WxVBlicm6VwyugXPDSgjDCWZvT4EplgJ06KzU0Btid+Raq1IZSzZSnuRfMBc0odywY9O5obqWlhlnqsPtKRWyOgZknIEuljAnPvgdfPLKijBWceZc859qqYMDoQI3TAYhR4AP1hBCNNciF0pp4iaWQxDkjfUmZIMJoJbz4zru3TdtEj6X4d4+d+BwHAkJMQXEhJXCJTUSIk8IwijOZY2FzSUTJlBWo5GUwVmMi3GhBZQo7x+kp9klTxofvE6HSpclmnhlTEgmmlKXHSgTtMSdJvLdSSoEoSSGs8FpzrHdQpDRU6Jz/McD/zcUiHF4TRynRAotJNEirKFUO4SFgJLGGpGYArRwCV3ohiNLSWOedCXIOdcaMU59eMn1Jn+cGHm01HiZ6h7PVtR76HgcNdf91cLD6cvN2aZ9shwPbpSGbtDP4iqsB57dtnkfk7qe/Zol2j8vkeIpcv4y1fYBskm9t00CV1WBPGh/3wxZddu2IZDC0GTIaPEB3Qus/Y4qeLJa+rZAWdjZkT3HYZm0ISAk1ckuALks+Ev/84NphaOvZ3kU/4hcV+/gNUgDE+Ct+Dz5sM/Ga7U8bB/ln4CwL8YBb5vZocUL0JuWyhsRYXX0ixFm6vn73czZA07ddQvmXCWUEOYYDygcCO+OmsSutn6j0zf3yM11+vno/++raJ+yTsUl8RRVX5lTTCrpEoXhOLkn60Ivn1ZqI/Hy5nUW33o+40/eTYW5wMA3nyCokp8ZcvGDgbJUChe8slVI4M0QwhqzDXzKsre/aoqRHfYP3N0hIOadC0pcMmqo6pJMrplmO7MwVRRP5Pzu3bDsXAzJ8cQSuP3VqNiO4Wd1PLxL3nz6s3iwuzoTpzeIvy/vV1fvvBO9uJpMPH6/uVovp/JfTiJxGdK70qXa2esCRGLZ1qt79u6tlmt3jRWvAnp5ebj5t3i5N9rf72/Wh37AzsEAZLut+2n497GxncU77P72+eH2Rvf7x9Z8vDqrP7IALujyyAxoMqYmVyL5B10I/78at7bdpdGe2fMk4XYhJHpAqWCCBWgdeS08Nl0B9bpzIbWnywKw2SDxOcdwvXIUcCVPnqmTWsok9Xv3+6r+LmAqmQQoAAA=="
_RUNBOOK_B64 = "H4sIAAAAAAACCp1Yy3LbRhbd8yu6KoupchEgAL7AuGbByJSjsiLTEp2kZkM00A2xI7wCNCTR5cV8xHzhfMmc2w2AlOxUzUwqCUUQ3X0f5557bv/AtnWZV5r57AO/v88kk88yabUqC1a3RVyWD6PRDz+wm1JL++VzI1l0UZfVRVnt7WJ/f+st95dlWwhOK/ef1vvrMnmQwlXVsYgj9qT0gfH+iN2Mvd9+ZrwQ7KrQsi6kZrLgcYYFbHeQrOhOY6nUyUE2TOOhyvNW00uslo3kdXJgMYcxSZnnSrOIz9KVWMipmPnL2AtmMonTOJ0Fs2UiwtkyDP3VdB5MF9GY8arKVLetzGMphBTs09rJjNGnkJSPss74cYwziqbNuxX6qWRcaw7LRO8SHIctuhkzLFFpt7mq2YE3cGDMVNFonmV2h0oVBdYiwFrlckx/NMOpYxOYuFWZsG+nquAZHhQiky4ywf5xtWVtlZVcsLLuj2aifCrMM9UgQn+2qkY4R6MX8bTJlc2Zi0V2dNkVLcdjvMgqWadlnfevBCwvYwUXc3Vfm/TSobw4siTjKnequhRtoop7tqt58uBcsOpQFnhd8qatZS4L7RoIdZFSRdXqZjRamwiSg40NqXxWjaZ9XoWU6dKEoXfix9HIYVHNC563B57nXPBDLsVikgCUSVk5Jl7qixTOY+j4vhN4wcLxI/bvf/4LIOL30om5yfMlvej86jPe6kNZK320QI3MDvucFyqVjXaT5hGgieBv0+x1uVfi2f2jKYvIpqriFGoWdQZPdM1VMYnMb8PDR57hka6lbN6S/Q2P+QG/xcDwpJGJ84d0am/pJMWz1g7wpuWzdhrfCVfeLPY9TwTB1OFe58edzwC/5KEqVaEJnhpnUvCiRmYy0Uj9vUf/+EvPFckylTyQoYhXfpz6LtbpqMPGCfZVxotxn20qrUwhj7wVSltHh7qzMCIY1GcFhO9JVhKw4+OLjL2lb0fzMgGMC+wIHAHULzDRGZTwoixUgl9PaVE9BZxlr+JIlSwEea2KEyeZrO4f/YmxfPKdXFqSESghHKPlt0v3F5vb3dXl1cV6d/XxZn+72X683d3Z9JEtAqGJJapBZkcEI8la+D9m8lEW7OmA/6khJw2CoaWjBOqgcwpWoaYz2Vg7hmDSCpSQ7qu3+S4Q34JH+5I/yzvVMYsM9PD15RIymQJvyKQuv8DAAfom9Ca7TyoDyxC7/mEBZNMBttCsQai02UaiQGQ9EAz2BQqSRDYNkfdVyux7FWEFyERi6NSBLodzOxfGloP7EIAAsiHvXxABMqysJMJoHOzdYnX5hMhKwO2evAdFRO/usK+zu11f3USExe7Br+vrHuovi6YPorFWPSKZY9ZYqulgCb/Q0zTRs4nxOaZZ3sKMA3+kgIF0bQyMIf9/ab/qfkI1CbWg5hS/gRTRDZCGaPJgTJ2YNEZUePKZIHT389oJ5gtmcaePxrWcvHEE8pYg2EcGBkWIRWnAkeMMWSNSt31qu7XoZaDcr+gRMmdfh52/jr46jmP+w48RkYebiwhvREnAp/NZEnvpSs696XQ5C4SM8Sj202QWxLOVH/oyDMNZOo8XIhXSn4lAzD0/5PNV4kW0+atsYdvXNJYmXjBN4rlcroI4mMaL5WK+kkImQSB9jy8TX8wW8czzPRFPV8uZ3XZA4IAlbB2LOAwC30+SJPDmK38+XUJASO4vQ+4vQi9cTfkinoYSv8+mKZezcBZzvvJFnC5CGditL6hBYN+Ktpwt0mXo+8tgkfgr8losA38aByLEprG/4nPuhwH34+Vimi48vsJ5QeLJebpYhKHottx6HWGQ/0uAbCYX/hw+C+F5aSwWi3hJa73ZPIC8ScJEyNUsjj0Rhgsv9IJY+qlIvGUi53bHDdXariQaXxeiLpVgvjt1fbZe39IpfIVTkJ05X6ZhPJsn3kJSKiWfrkIZJwsYLmbLmZhPg2VI4QrxihdDfk2Xwl/SKabamkQRflKVTDqhw7ZHFHaBKgeDm7qzOqhr8B01Ie+yeFR1WZB2+LFfBBMD93mMr9b4gL77tmFffH63Zv7U9cASbCebjENjglroxUfVkGbx3GDpQluduW/cHjPzZf0Rb/hL1xuT3Q9KOxlaXYF3zKotaKl8YjhxSq/ctPn2CAvmbmA741PNSU3RDlOjpuBcp/jg36tafSrrB9BJr1f2VSei0T2I3i35/K1h91kZo2F0/p/FxBIxiAEVLUhsgnl/kglvG3la3fUGvISGSlRipHMEKoUqq1QVjV/yWQIa070mPjvLKiLHoc+y1Q4tNT43PKX210gpKJMVHDaJ5PrFemQ1N5t2brPm2BCTYIWJlKZ+aQRwRJuYkrcHwGxDZg5RIHI4UXlV1hCodYk2FUuoVEmNJs3U/UFb7uwQZfNRAWYQDLBUq8Zo8iE6qrBNDmytNBDR6DJHw4lM1uqWLLdat3PbRvrYd4EuNWd+uuyubOuEugGo8MiQCxtLNLJ7I+ChAJWdjyClkXvoGAw3rMGxrEyZv3AhuTtnWMpV1vQ+ymfqgOrRxpCpdGhQTaPuETkoApWmsqZ4Y7Ia6i3luSJ5/xvFeLv7RMDpBq3xKXKMZ2gOneynzZocuCVp5ejSwQeWBhsCVJeIMfs0efeJmd6E+I7Z7zc32/XFB4YawfRT3KOI8Eki3Pg77icdh+YTWzE464nXoksm9d2yzJIDEIsUYKSDulMQop34E1LbdHURocCe4glsCkmsci9JRH4oMAmdybtE1oaGbOxRcYQERPx76sqK9m+VvhGvB04jmDDJcSB1IXl+fKVfjPi2iW8QiEyaAdAxtppBgRG8Et3WCPGRxTQzE/+h+LJXE8S3A0TE+t6tjKMYqjb9tD4a+S67MDVM4oc002ngPs3TVIz/6/TujgKXrQWCX4IKBgXCY+QfOKeaePOGfr8iEfLmDUrBhAHn91w8zPtwvDgN/WXhjsCXG8NJ31NeRF1GXpmp2aq0v5BxJGaENcodzVx22yIAgHEiaepWNLWChS1LpKpG37e/mJFlYqb246TjbFKgJQQRCfbs+Bbx63SynTnQjxs7zYD7SCjPXXYtSQpGm5v1T9eb/d3VzXt83H78vNvsqfD+jum4Bb3osp/BO5fKpkKoUNtAQ9cAYY9Tl/QKrRSSatwdgR7eWaVG57PL7TSAOWY60geo7UNJ9wVIPS0iNVppXhAftRnp6BQh764k8HKb9e5bBdvigCUd0F0hRNvbj79sd/7+w/o9ObL5fXPx2cxCP32+eXe9cb9QFxio3d5Q0IbuKET6ac9vwdexB8XAjAf14/mlDq1m8LofR857SFbeE97vNK+7hsLxAZ/7M0C2ZmszfBkGNjcSQA+iAsdf20AwMNckOeDSd1qn7xegoQJUAiUHWiFJzetMmaHHigk7lmSyK/SkrOuOncpSo74hAK0N0t6TEFiph9BMLUiimHYI5zCu56bS4xq5OtiSvsZzpx85LKn10zf6ktVWaZtl3xCggYS5VLvZhT3lAiMNtSLSBnRsp/mcnmIzyt4z2U7dyPGBa5AljRAN7M652ZDKECydnVmGqn2kASExA/ETcKhlYc2lkNk//jvhQ+PZb0Zl3G7uPv+y2V8Ae9eb3ebd/m63fr+566sHfUR+dxZrHlRlGpc51UyKA+BP5Hx+M4fc1g/SEHbqEEWPWdLWpoEO79vHXVHaKwq8BV1YsPdmwKd7R9NHzTRMQqLLV39dYlpyQrTUB9VeB4JO6c0MrVPzHJ2X+jF1FfrEybYDAmDdhvZKD66zmpvxGyoLwcb7hYav6CiNHdi3f9nTtQISz1r6SQdP+u4NmQF5NWaXGdc/taQnmr/o4+zbPj5cG3ellXB7c8vP+3k2EHyU4pAEGUXFysw4XXCiweGIs06f8RoxQJfqTMRvkFRdBzTzXEccXXGYJ3RJ2t/B2CvDntMuP4LD7M3Oz+ubdx8vL22Pp3s5Iv+tN7nEv75ziQN7kJuKR1c0t3cddQs0cqo+UsLg6Sorj4au7FVt/2IjQdP2qkg1JdyWppZ1rQxnWFR24uS00o7xPW4mqbmB7ZSK5YNXI76Sg2l2bY/j8VAnHSzttT4xgkmfWYKHtNpSSA9W07Dxo+GT4akpL2hQI26LYWyzHcuyjYkkQwHmJhVmPcGbuvxJjUlL19SvSiJXTX4gCEb7Irm/ngQzGoCi6tOEBPqjxjR2crwjqUzF3Q31WaRqeR7tF/m0LZzmKCMKRT/BnCe3v3d/DS1TkaSyXorS4RI+U4UZcdGXnwoiS1kfWU53pTU9Nj2hiz83oxTIUv3ZSiMxxuzi9sLyu7m7oQHAmG83aA6qGpt7RWZk0vh7eAAF3hP2NYsur27W1/vLKzTyX9Y3V5ebu51FvDv6DwtAlYIAGgAA"
def _inflate(value): return gzip.decompress(base64.b64decode(value)).decode("utf-8")
def _run(command):
    print("RUN:", " ".join(map(str, command)), flush=True); subprocess.check_call([str(x) for x in command])
def _metadata_files(root, predicate):
    skip={"train","val","test",".git","__pycache__",".ipynb_checkpoints"}; found=[]; visited=set()
    for current,dirs,files in os.walk(root,followlinks=True):
        real=Path(current).resolve()
        if real in visited: dirs[:]=[]; continue
        visited.add(real); dirs[:]=[n for n in dirs if n not in skip]
        found.extend(Path(current)/n for n in files if predicate(n))
    return sorted(found)
KAGGLE_INPUT_ROOT=Path("/kaggle/input")
if not KAGGLE_INPUT_ROOT.is_dir(): raise RuntimeError(f"/kaggle/input is missing. Attach {FINAL_V1_DATASET_SLUG} and {CHECKPOINT_DATASET_SLUG}.")
attached_roots=sorted(str(p) for p in KAGGLE_INPUT_ROOT.iterdir())
print("Attached Kaggle input roots:", attached_roots)
if CLEAN_START:
    for target in (BOOTSTRAP_ROOT,INPUT_ROOT,WORK_ROOT):
        if target.exists(): shutil.rmtree(target)
BOOTSTRAP_ROOT.mkdir(parents=True,exist_ok=True); INPUT_ROOT.mkdir(parents=True,exist_ok=True)
def _repo_contains_base():
    if not (RESEARCH_ROOT/".git").is_dir(): return False
    return subprocess.run(["git","-C",str(RESEARCH_ROOT),"merge-base","--is-ancestor",RESEARCH_BASE_COMMIT,"HEAD"],check=False,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode==0
if not _repo_contains_base():
    if RESEARCH_ROOT.exists(): shutil.rmtree(RESEARCH_ROOT)
    RESEARCH_ROOT.mkdir(parents=True)
    _run(["git","-C",str(RESEARCH_ROOT),"init","-q"]); _run(["git","-C",str(RESEARCH_ROOT),"remote","add","origin",RESEARCH_REPO_URL])
    _run(["git","-C",str(RESEARCH_ROOT),"fetch","--depth","1","origin",RESEARCH_BASE_COMMIT]); _run(["git","-C",str(RESEARCH_ROOT),"checkout","-b","prompt1-kaggle-runtime","FETCH_HEAD"])
if subprocess.check_output(["git","-C",str(RESEARCH_ROOT),"rev-parse",RESEARCH_BASE_COMMIT],text=True).strip()!=RESEARCH_BASE_COMMIT: raise RuntimeError("Pinned research base commit is unavailable")
overlays={
    "journal_extension/scripts/run_prompt1_kaggle.py":_EXECUTOR_B64,
    "journal_extension/track_c_r07/shared_foundation/TRACKBC_R07_AUTHORITY_BINDING_v1.json":_AUTHORITY_BINDING_B64,
    "journal_extension/track_c_r07/shared_foundation/R07_DEPLOYMENT_RECONSTRUCTION_CONTRACT_v1.json":_RECONSTRUCTION_CONTRACT_B64,
    "journal_extension/kaggle/PROMPT1_KAGGLE_RUNBOOK.md":_RUNBOOK_B64,
}
for relative,payload in overlays.items():
    target=RESEARCH_ROOT/relative; target.parent.mkdir(parents=True,exist_ok=True); target.write_text(_inflate(payload),encoding="utf-8")
(INPUT_ROOT/"plan.md").write_text(_inflate(_PLAN_B64),encoding="utf-8")
(INPUT_ROOT/"R07_PRODUCTION_BASELINE_AUDIT.md").write_text(_inflate(_BASELINE_B64),encoding="utf-8")
if subprocess.check_output(["git","-C",str(RESEARCH_ROOT),"status","--porcelain=v1"],text=True).strip():
    _run(["git","-C",str(RESEARCH_ROOT),"config","user.email","cropcop-prompt1@localhost"]); _run(["git","-C",str(RESEARCH_ROOT),"config","user.name","CropCop Prompt 1"])
    _run(["git","-C",str(RESEARCH_ROOT),"add",*overlays.keys()]); _run(["git","-C",str(RESEARCH_ROOT),"commit","-m","kaggle: bind QA-locked Prompt 1 overlay"])
# Inputs are attached by Kaggle; discovery is hash-based so dataset mount names do not matter.
final_download=KAGGLE_INPUT_ROOT; checkpoint_download=KAGGLE_INPUT_ROOT
def _hash_match(root,basename,expected):
    matches=[p for p in _metadata_files(root,lambda name:name==basename) if hashlib.sha256(p.read_bytes()).hexdigest()==expected]
    if len(matches)!=1: raise RuntimeError(f"Expected exactly one hash-valid {basename} under attached Kaggle inputs; found {matches}; attached={attached_roots}")
    return matches[0]
def _find_final_authority(root):
    manifests=[p.resolve() for p in _metadata_files(root,lambda n:n=="final_manifest.csv") if "CropCop_Final_v1_CERTIFICATION_REPORTS" not in p.resolve().parts and hashlib.sha256(p.read_bytes()).hexdigest()=="bdb82211ccc2059153724eea178a1680893a6b38ecc243fae484baa91dbf68e2"]
    class_maps=[p.resolve() for p in _metadata_files(root,lambda n:n=="class_to_idx.json") if "CropCop_Final_v1_CERTIFICATION_REPORTS" not in p.resolve().parts and hashlib.sha256(p.read_bytes()).hexdigest()=="46f7811726c19c42bd7213b2d8178b19a5a182a1b763f60a94ee2c0e5f6688d2"]
    candidates=[]
    for m in manifests:
        base=m.parent.parent.resolve()
        if base.name!="CropCop_Final_v1" or not (base/"dataset/train").is_dir() or not (base/"dataset/val").is_dir(): continue
        for c in class_maps:
            if c.parent.resolve()==m.parent.resolve(): candidates.append((m,c,base))
    candidates=list(dict.fromkeys(candidates))
    if len(candidates)!=1: raise RuntimeError(f"Expected one canonical image-backed CropCop_Final_v1 authority from {FINAL_V1_DATASET_SLUG}; certification-report copies are excluded; found {candidates}; attached={attached_roots}")
    return candidates[0]
manifest,class_map,final_root=_find_final_authority(final_download)
final_link=INPUT_ROOT/"final_v1"
if final_link.is_symlink():
    if final_link.resolve()!=final_root: raise RuntimeError(f"Existing final_v1 link points to {final_link.resolve()} not {final_root}")
elif final_link.exists():
    raise RuntimeError(f"Refusing to overwrite existing non-link {final_link}")
else:
    final_link.symlink_to(final_root,target_is_directory=True)
checkpoint=_hash_match(checkpoint_download,"selected.g00000170.dc7fea2e8db91bf1.ckpt","dc7fea2e8db91bf1fc023cb5e792b23b67659edec22e10a7c1d46b4010db3974"); shutil.copy2(checkpoint,INPUT_ROOT/checkpoint.name)
print("Standalone bootstrap complete."); print("Research checkout:",RESEARCH_ROOT); print("Final-V1 authority:",final_root); print("Checkpoint:",INPUT_ROOT/checkpoint.name); print("INPUT_ROOT:",INPUT_ROOT)



In [ ]:
# Install Prompt 1 into an isolated venv; never mutate Kaggle's global environment.
import json, shutil, subprocess, sys
RUNTIME_ROOT = Path("/kaggle/working/cropcop_prompt1_venv")
RUNTIME_PYTHON = RUNTIME_ROOT / "bin/python"
packages = ["torch==2.12.1", "torchvision==0.27.1", "executorch==1.3.1", "torchao==0.17.0", "scikit-learn==1.7.1", "pillow==12.3.0", "numpy==2.5.2", "wrapt==1.17.3"]
if sys.version_info[:2] != (3,12): raise RuntimeError(f"Kaggle Python 3.12.x required before install; observed {sys.version}")
gpu_names = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).splitlines()
if not any("T4" in name for name in gpu_names): raise RuntimeError(f"Tesla T4 required before install; observed {gpu_names}")
free_bytes = shutil.disk_usage("/kaggle/working").free
if free_bytes < 8 * 1024**3: raise RuntimeError(f"At least 8 GiB free in /kaggle/working is required; observed {free_bytes/1024**3:.2f} GiB")
if CLEAN_START and RUNTIME_ROOT.exists(): shutil.rmtree(RUNTIME_ROOT)
if not RUNTIME_PYTHON.is_file(): subprocess.check_call([sys.executable, "-m", "venv", "--without-pip", str(RUNTIME_ROOT)])
RUNTIME_SITE = Path(subprocess.check_output([str(RUNTIME_PYTHON), "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"], text=True).strip()).resolve()
if RUNTIME_ROOT.resolve() not in RUNTIME_SITE.parents: raise RuntimeError(f"Venv site-packages escaped runtime root: {RUNTIME_SITE}")
pip_ready = subprocess.run([str(RUNTIME_PYTHON), "-m", "pip", "--version"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not pip_ready:
    RUNTIME_SITE.mkdir(parents=True, exist_ok=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "--upgrade", "--target", str(RUNTIME_SITE), "pip"])
probe = "import importlib.metadata as m; expected=" + repr({p.split("==")[0]:p.split("==")[1] for p in packages}) + "; observed={k:m.version(k) for k in expected}; assert observed==expected, (observed,expected); import torch,torchvision,executorch,torchao,numpy,sklearn,PIL"
ready = subprocess.run([str(RUNTIME_PYTHON), "-c", probe], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not ready:
    subprocess.check_call([str(RUNTIME_PYTHON), "-m", "pip", "install", "--no-cache-dir", *packages])
subprocess.check_call([str(RUNTIME_PYTHON), "-m", "pip", "check"])
subprocess.check_call([str(RUNTIME_PYTHON), "-c", probe])
if shutil.which("flatc") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "flatbuffers-compiler"])
if not PIPELINE.is_file(): raise RuntimeError(f"Executor missing after bootstrap: {PIPELINE}")
print("Isolated pinned runtime and toolchain setup complete:", RUNTIME_PYTHON)


In [ ]:
# Shared stage runner with explicit PASS-marker resume behavior.
import subprocess, sys

def marker_passes(relative):
    if not (WORK_ROOT / relative).is_file():
        return False
    command = [str(RUNTIME_PYTHON), str(PIPELINE), 'verify', '--input-root', str(INPUT_ROOT), '--work-root', str(WORK_ROOT), '--marker', relative]
    completed = subprocess.run(command, text=True, capture_output=True)
    if completed.returncode == 0:
        return True
    if (WORK_ROOT / relative).exists():
        print(f'RERUN: stale or invalid resume marker {relative}: {completed.stderr[-800:]}')
    return False

def run_stage(stage, marker=None, extra=()):
    if RESUME_COMPLETED_STAGES and marker and marker_passes(marker):
        print(f'SKIP {stage}: verified PASS marker {marker}')
        return
    command = [str(RUNTIME_PYTHON), str(PIPELINE), stage, '--input-root', str(INPUT_ROOT), '--work-root', str(WORK_ROOT), '--workers', str(NUM_WORKERS), '--source-batch-size', str(SOURCE_BATCH_SIZE), *extra]
    print('RUN', ' '.join(command))
    subprocess.check_call(command)


## P0/F0 binding and complete input preflight

In [ ]:
preflight_extra = () if ENABLE_SINGLE_ROUTE_PTQ else ('--skip-ptq',)
run_stage('preflight', 'evidence/PROMPT1_PREFLIGHT_v1.json', preflight_extra)


## F1 â€” exact checkpoint reconstruction and full DS-V1-VAL replay

In [ ]:
run_stage('source', 'evidence/R07_S1_VALIDATION_REPLAY_REPORT_v1.json')


## F2 â€” version-matched FP32 ExecuTorch/XNNPACK export, load and forward

In [ ]:
run_stage('fp32', 'evidence/TRACKC_R07_RUNTIME_FEASIBILITY_REPORT_v1.json')


## F3 â€” prospective FP32 sourceâ†’artifact lock and validation-wide raw-logit parity

The executor writes and seals the numerical policy before it opens full parity results. This is intentionally a separate cell and can be long-running because the production artifact has a fixed batch-1 Android input schema.

In [ ]:
run_stage('parity', 'evidence/R07_FP32_SOURCE_ARTIFACT_FIDELITY_v1.json')


## F4 â€” optional single-route PTQ

Exactly one static PT2E/XNNPACK route is allowed. A technical or retention failure produces a terminal certificate and continues with FP32; there is no tuning sweep.

In [ ]:
if ENABLE_SINGLE_ROUTE_PTQ:
    if RESUME_COMPLETED_STAGES and (marker_passes('evidence/R07_INT8_RESULT_v1.json') or marker_passes('evidence/R07_INT8_FAILURE_CERTIFICATE_v1.json')):
        print('SKIP ptq: terminal result already exists')
    else:
        run_stage('ptq')
else:
    print('PTQ prospectively disabled; seal cell will record the absent branch.')


## F5â€“F7 â€” representation selection, deployment bundles, publication scan, device policy and final handoff

In [ ]:
extra = () if ENABLE_SINGLE_ROUTE_PTQ else ('--skip-ptq',)
run_stage('seal', 'PROMPT1_FOUNDATION_HANDOFF.json', extra)


## Download the complete Prompt 1 evidence bundle

This final cell verifies the bundle hash, displays the final gate summary, and emits a clickable Kaggle download link.

In [ ]:
import hashlib, json
from IPython.display import FileLink, display

subprocess.check_call([str(RUNTIME_PYTHON), str(PIPELINE), 'verify', '--input-root', str(INPUT_ROOT), '--work-root', str(WORK_ROOT), '--marker', 'PROMPT1_FOUNDATION_HANDOFF.json'])
download_record = json.loads((WORK_ROOT / 'FINAL_DOWNLOAD.json').read_text())
bundle = Path(download_record['path'])
hasher = hashlib.sha256()
with bundle.open('rb') as handle:
    for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
        hasher.update(chunk)
digest = hasher.hexdigest()
assert digest == download_record['sha256']
handoff = json.loads((WORK_ROOT / 'PROMPT1_FOUNDATION_HANDOFF.json').read_text())
assert handoff['status'] == 'PASS' and handoff['next_prompt_authorized'] is True
print(json.dumps({'gates': handoff['gates'], 'bundle_bytes': bundle.stat().st_size, 'bundle_sha256': digest}, indent=2))
display(FileLink(str(bundle), result_html_prefix='<b>Download Prompt 1 evidence bundle:</b> '))
